<h1>Task 1: Data Cleaning and Preprocessing</h1>
Description: Work with a raw dataset (e.g., CSV file) that contains missing values, duplicates, and inconsistent data formats.
<h3>Objectives:</h3>
<ul>
    <li>Load the dataset using pandas.</li>
    <li>Identify and handle missing values (e.g., imputation or removal).</li>
    <li>Remove duplicate rows and standardize inconsistent data formats (e.g., date formats, categorical variables).</li>
</ul>
<p>Tools: Python, pandas.</p> 

In [ ]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root() -> Path:
    """To Find the project directory from the notebook's current working directory."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Project root not found: expected directories 'src' and 'data/raw'."
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src import quality_report
importlib.reload(quality_report)
from src import seek_duplicate
importlib.reload(seek_duplicate)
from src.seek_duplicate import seek_duplicate_rows
from src import strip_string
from src import correct_ohlc

RAW_DIR = PROJECT_ROOT / "data/raw"
CLEAN_DIR = PROJECT_ROOT / "data/cleaned_data"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projet : {PROJECT_ROOT}")
print(f"Fichiers bruts : {len(list(RAW_DIR.rglob('*.csv')))}")

Projet : /home/broman/Downloads/CodVeva_internship/codveda_datanalysis
Fichiers bruts : 6


# 1.Iris cleaning

In [5]:
iris = pd.read_csv(RAW_DIR / "1) iris.csv")
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [6]:
iris.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 7.2 KB


In [7]:
# We drop duplicate rows
iris = iris.drop_duplicates()

In [8]:
# Search any duplicate rows
seek_duplicate_rows(iris)

# We verify if the data is well cleaned
quality_report.rep_quality(iris,'cleaning report')

Number of duplicate rows: 0

--- cleaning report ---
Dimensions : (147, 5)
Duplicate rows : 0
No missing values.


Cleaning processing

In [9]:
# We clean blank for the string values
strip_string.strip_string_columns(iris)

# We verify if the data is well cleaned
quality_report.rep_quality(iris,'cleaning report')

# We save the cleaned data  
iris.to_csv(CLEAN_DIR / "c_iris.csv", index=False)


--- cleaning report ---
Dimensions : (147, 5)
Duplicate rows : 0
No missing values.


# 2. Stock prices cleaning

In [10]:
stock_price = pd.read_csv(RAW_DIR / "2) Stock Prices Data Set.csv")
stock_price.head()

,symbol,date,open,high,low,close,volume
0,AAL,2014-01-02,25.0700,25.8200,25.0600,25.3600,8998943
1,AAPL,2014-01-02,79.3828,79.5756,78.8601,79.0185,58791957
2,AAP,2014-01-02,110.3600,111.8800,109.2900,109.7400,542711
3,ABBV,2014-01-02,52.1200,52.3300,51.5200,51.9800,4569061
4,ABC,2014-01-02,70.1100,70.2300,69.4800,69.8900,1148391


In [9]:
stock_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   symbol  497472 non-null  str    
 1   date    497472 non-null  str    
 2   open    497461 non-null  float64
 3   high    497464 non-null  float64
 4   low     497464 non-null  float64
 5   close   497472 non-null  float64
 6   volume  497472 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 26.6 MB


In [10]:
# Search any duplicate rows
seek_duplicate_rows(stock_price)

# We verify if the data is well cleaned
quality_report.rep_quality(stock_price,'cleaning report')

Number of duplicate rows: 0

--- cleaning report ---
Dimensions : (497472, 7)
Duplicate rows : 0
Missing values:
open    11
high     8
low      8
dtype: int64


In [11]:
# Detect : Identifying and marking the inconsistent lines
stock_price['Consistent'] = (stock_price['high'] >= stock_price[['open', 'low', 'close']].max(axis=1)) & \
                 (stock_price['low'] <= stock_price[['open', 'high', 'close']].min(axis=1))

#Print the count of all the rows with the inconsistent column  
print("Count of consistent rows and the inconsistent one:", stock_price['Consistent'].value_counts())


Count of consistent rows and the inconsistent one: Consistent
True     497452
False        20
Name: count, dtype: int64


<h3>> Cleaning processing</h3>

In [94]:
# We clean blank for the string values
strip_string.strip_string_columns(stock_price)

# We drop duplicate rows
stock_price = stock_price.drop_duplicates()

# Format the data type
stock_price["date"] = pd.to_datetime(stock_price["date"], errors="coerce")

# Reload the module when the function has been edited during this notebook session
importlib.reload(correct_ohlc)
stock_price = correct_ohlc.ohlc_correct(stock_price)

# Recompute the consistency flag after the correction
stock_price["Consistent"] = (
    (stock_price["high"] >= stock_price[["open", "low", "close"]].max(axis=1))
    & (stock_price["low"] <= stock_price[["open", "high", "close"]].min(axis=1))
)

print("Missing OHLC values after correction:")
print(stock_price[["open", "high", "low", "close"]].isna().sum())
print("Inconsistent rows after correction:", (~stock_price["Consistent"]).sum())

#Remove the column consistent from the dataframe
stock_price.drop(columns=['Consistent'], inplace=True)

# Save the cleaned stock prices
stock_price.to_csv(CLEAN_DIR / "c_stock.csv", index=False)
print(f"Cleaned data saved to: {CLEAN_DIR / 'c_stock.csv'}")

✅ Success : All the OHLC data are structurally correct. No process.
Missing OHLC values after correction:
open     0
high     0
low      0
close    0
dtype: int64
Inconsistent rows after correction: 0
Cleaned data saved to: /home/broman/Downloads/CodVeva_internship/codveda_datanalysis/data/cleaned_data/c_stock.csv


<h3>> Verification<h3/>

In [95]:
quality_report.rep_quality(stock_price, 'stock price cleaning report')


--- stock price cleaning report ---
Dimensions : (497472, 7)
Duplicate rows : 0
No missing values.


In [96]:
stock_price.info()

<class 'pandas.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   symbol  497472 non-null  str           
 1   date    497472 non-null  datetime64[us]
 2   open    497472 non-null  float64       
 3   high    497472 non-null  float64       
 4   low     497472 non-null  float64       
 5   close   497472 non-null  float64       
 6   volume  497472 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 26.6 MB


# 3. Sentiment cleaning

In [48]:
sent = pd.read_csv(RAW_DIR/'3) Sentiment dataset.csv')

#fast overview
sent.head()

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19


In [49]:
sent.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0.1  732 non-null    int64  
 1   Unnamed: 0    732 non-null    int64  
 2   Text          732 non-null    str    
 3   Sentiment     732 non-null    str    
 4   Timestamp     732 non-null    str    
 5   User          732 non-null    str    
 6   Platform      732 non-null    str    
 7   Hashtags      732 non-null    str    
 8   Retweets      732 non-null    float64
 9   Likes         732 non-null    float64
 10  Country       732 non-null    str    
 11  Year          732 non-null    int64  
 12  Month         732 non-null    int64  
 13  Day           732 non-null    int64  
 14  Hour          732 non-null    int64  
dtypes: float64(2), int64(6), str(7)
memory usage: 85.9 KB


In [51]:
# Search any duplicate rows
seek_duplicate_rows(sent)

# We verify if the data is well cleaned
quality_report.rep_quality(sent,'sentiment cleaning report')

Number of duplicate rows: 0

--- sentiment cleaning report ---
Dimensions : (732, 15)
Duplicate rows : 0
No missing values.


Cleaning processing

In [57]:
# Remove automatically generated index columns
sent = sent.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], errors="ignore")

# Clean whitespace in text columns
strip_string.strip_string_columns(sent)

# Convert and validate the timestamp
sent["Timestamp"] = pd.to_datetime(sent["Timestamp"], errors="coerce")

# Normalize sentiment labels
sent["Sentiment"] = sent["Sentiment"].str.strip().str.lower()

#Drop the duplicate rows
sent = sent.drop_duplicates()
quality_report.rep_quality(sent,"sentiment check up")


--- sentiment check up ---
Dimensions : (711, 13)
Duplicate rows : 0
No missing values.


In [53]:
sent.info()

<class 'pandas.DataFrame'>
Index: 711 entries, 0 to 731
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Text       711 non-null    str           
 1   Sentiment  711 non-null    str           
 2   Timestamp  711 non-null    datetime64[us]
 3   User       711 non-null    str           
 4   Platform   711 non-null    str           
 5   Hashtags   711 non-null    str           
 6   Retweets   711 non-null    float64       
 7   Likes      711 non-null    float64       
 8   Country    711 non-null    str           
 9   Year       711 non-null    int64         
 10  Month      711 non-null    int64         
 11  Day        711 non-null    int64         
 12  Hour       711 non-null    int64         
dtypes: datetime64[us](1), float64(2), int64(4), str(6)
memory usage: 77.8 KB


In [54]:
sent.head()

,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,Enjoying a beautiful day at the park! ...,positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,Traffic was terrible this morning. ...,negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,Just finished an amazing workout! 💪 ...,positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,Excited about the upcoming weekend getaway! ...,positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,Trying out a new recipe for dinner tonight. ...,neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19


In [68]:
# Save the cleaned dataset
sent.to_csv(CLEAN_DIR / "c_sentiment.csv", index=False)

<h2>4. House prediction data cleaning</h2>

In [23]:

house = pd.read_csv(RAW_DIR / "4) house Prediction Data Set.csv", sep=r'\s+', header=None)

# Initial overview
house.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [24]:
house.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       506 non-null    float64
 1   1       506 non-null    float64
 2   2       506 non-null    float64
 3   3       506 non-null    int64  
 4   4       506 non-null    float64
 5   5       506 non-null    float64
 6   6       506 non-null    float64
 7   7       506 non-null    float64
 8   8       506 non-null    int64  
 9   9       506 non-null    float64
 10  10      506 non-null    float64
 11  11      506 non-null    float64
 12  12      506 non-null    float64
 13  13      506 non-null    float64
dtypes: float64(12), int64(2)
memory usage: 55.5 KB


In [25]:
quality_report.rep_quality(house, "house cleaning quality report")


--- house cleaning quality report ---
Dimensions : (506, 14)
Duplicate rows : 0
No missing values.


In [76]:
# Remove automatically generated index columns
house = house.loc[:, ~house.columns.astype(str).str.startswith("Unnamed:")]
house = house.reset_index(drop=True)

quality_report.rep_quality(house, "house prediction cleaning report")

# Save the cleaned dataset without the DataFrame index
house.to_csv(CLEAN_DIR / "c_house.csv", index=False)

print(f"Cleaned data saved to: {CLEAN_DIR / 'c_house.csv'}")
house.head()


--- house prediction cleaning report ---
Dimensions : (506, 14)
Duplicate rows : 0
No missing values.
Cleaned data saved to: /home/broman/Downloads/CodVeva_internship/codveda_datanalysis/data/cleaned_data/c_house.csv


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


<h2>5. Churn data cleaning</h2>

In [78]:
churn_1 = pd.read_csv(RAW_DIR / "churn_prediction_data/churn-bigml-20.csv")
churn_2 = pd.read_csv(RAW_DIR / "churn_prediction_data/churn-bigml-80.csv")

In [83]:
churn_1.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,LA,117,408,No,No,0,184.5,97,31.37,351.6,80,29.89,215.8,90,9.71,8.7,4,2.35,1,False
1,IN,65,415,No,No,0,129.1,137,21.95,228.5,83,19.42,208.8,111,9.40,12.7,6,3.43,4,True
2,NY,161,415,No,No,0,332.9,67,56.59,317.8,97,27.01,160.6,128,7.23,5.4,9,1.46,4,True
3,SC,111,415,No,No,0,110.4,103,18.77,137.3,102,11.67,189.6,105,8.53,7.7,6,2.08,2,False
4,HI,49,510,No,No,0,119.3,117,20.28,215.1,109,18.28,178.7,90,8.04,11.1,1,3.00,1,False


In [84]:
churn_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 667 entries, 0 to 666
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   667 non-null    str    
 1   Account length          667 non-null    int64  
 2   Area code               667 non-null    int64  
 3   International plan      667 non-null    str    
 4   Voice mail plan         667 non-null    str    
 5   Number vmail messages   667 non-null    int64  
 6   Total day minutes       667 non-null    float64
 7   Total day calls         667 non-null    int64  
 8   Total day charge        667 non-null    float64
 9   Total eve minutes       667 non-null    float64
 10  Total eve calls         667 non-null    int64  
 11  Total eve charge        667 non-null    float64
 12  Total night minutes     667 non-null    float64
 13  Total night calls       667 non-null    int64  
 14  Total night charge      667 non-null    float64
 15  

In [86]:
quality_report.rep_quality(churn_1, "churn prediction cleaning report")


--- churn prediction cleaning report ---
Dimensions : (667, 20)
Duplicate rows : 0
No missing values.


In [85]:
churn_2.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [87]:
churn_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   str    
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   str    
 4   Voice mail plan         2666 non-null   str    
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge      2666 non-null   float64
 15

In [88]:
quality_report.rep_quality(churn_2, "churn 2 prediction cleaning report")


--- churn 2 prediction cleaning report ---
Dimensions : (2666, 20)
Duplicate rows : 0
No missing values.


In [89]:
# We assembly churn_1 and churn_2 into a single DataFrame
churn_combined = pd.concat([churn_1, churn_2], ignore_index=True)

In [90]:
churn_combined.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,LA,117,408,No,No,0,184.5,97,31.37,351.6,80,29.89,215.8,90,9.71,8.7,4,2.35,1,False
1,IN,65,415,No,No,0,129.1,137,21.95,228.5,83,19.42,208.8,111,9.40,12.7,6,3.43,4,True
2,NY,161,415,No,No,0,332.9,67,56.59,317.8,97,27.01,160.6,128,7.23,5.4,9,1.46,4,True
3,SC,111,415,No,No,0,110.4,103,18.77,137.3,102,11.67,189.6,105,8.53,7.7,6,2.08,2,False
4,HI,49,510,No,No,0,119.3,117,20.28,215.1,109,18.28,178.7,90,8.04,11.1,1,3.00,1,False


In [91]:
churn_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   str    
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   str    
 4   Voice mail plan         3333 non-null   str    
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   float64
 15

In [92]:
quality_report.rep_quality(churn_combined, "churn prediction cleaning report")


--- churn prediction cleaning report ---
Dimensions : (3333, 20)
Duplicate rows : 0
No missing values.


In [93]:
# Save the cleaned datasets without the DataFrame index
churn_combined.to_csv(CLEAN_DIR / "c_churn_combined.csv", index=False)